In [7]:
!mkdir -p src

In [2]:
%%writefile src/app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import joblib
import re
import string
import os

Overwriting src/app.py


####Import Libraries

In [3]:
import streamlit as st
import pandas as pd
import plotly.express as px
import joblib
import re
import string
import os

####Page Configuration

In [4]:
st.set_page_config(
    page_title="ShopEase Europe",
    page_icon="🛒",
    layout="wide"
)

2026-07-09 18:47:42.097 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


####Load Model

In [9]:
model = joblib.load("/content/best_sentiment_model.pkl")

####Load Dashboard Data

In [10]:
df = pd.read_csv(
    "/content/shopease_powerbi_clean (1).csv"
)

####Text Cleaning Function

In [11]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"\S+@\S+", "", text)

    text = re.sub(r"\d+", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

####Driver Function

In [12]:
driver_keywords = {

"Delivery":[
"delivery",
"driver",
"parcel",
"shipping",
"late"
],

"Product Quality":[
"quality",
"broken",
"faulty",
"damaged"
],

"Customer Support":[
"support",
"service",
"agent",
"staff"
],

"Price":[
"refund",
"cheap",
"expensive",
"price"
],

"Website":[
"website",
"app",
"payment",
"checkout"
]

}

In [13]:
def detect_driver(text):

    text = text.lower()

    for driver, words in driver_keywords.items():

        for word in words:

            if word in text:

                return driver

    return "Other"

####Sidebar

In [14]:
page = st.sidebar.radio(

"Navigation",

[
"🏠 Home",
"🔍 Single Prediction",
"📂 Batch Prediction",
"📊 Dashboard",
"📈 Model Performance",
"ℹ️ About"
]

)

2026-07-09 18:49:43.588 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:43.589 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:43.590 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:43.590 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-07-09 18:49:43.591 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:43.718 
  command:

    streamlit run /usr/local/lib/python3.12/

####Home Page

In [15]:
if page=="🏠 Home":

    st.title("🛒 ShopEase Europe")

    st.markdown("""

### Customer Feedback Intelligence Platform

This application uses Machine Learning
to analyse customer reviews.

### Features

✔ Predict customer sentiment

✔ Batch prediction

✔ Dashboard

✔ Business insights

✔ Power BI integration

""")

2026-07-09 18:49:56.358 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:56.359 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:56.359 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:56.360 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:56.361 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:49:56.361 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


####Single Prediction

In [16]:
if page=="🔍 Single Prediction":

    st.header("Predict Customer Sentiment")

    review = st.text_area(

        "Customer Review"

    )

#####Predict

In [20]:
if st.button("Predict"):

    clean = clean_text(review)

    prediction = model.predict([clean])[0]

    driver = detect_driver(clean)

    st.success(

        f"Predicted Sentiment : {prediction}"

    )

    st.info(

        f"Main Driver : {driver}"

    )

2026-07-09 18:52:01.913 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:52:01.914 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:52:01.916 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:52:01.917 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 18:52:01.919 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


#####Batch Prediction

In [33]:
if page=="📂 Batch Prediction":

    file = st.file_uploader(

        "Upload CSV",

        type=["csv"]

    )

    if file:

        upload = pd.read_csv(file)

        upload["clean_review"] = (

            upload["review"]

            .apply(clean_text)

        )

        upload["prediction"] = (

            model.predict(

                upload["clean_review"]

            )

        )

        upload["driver"] = (

            upload["clean_review"]

            .apply(detect_driver)

        )
        st.dataframe(upload)

        st.download_button(

        "Download Predictions",

        upload.to_csv(index=False),

        "predictions.csv"

        )

#####DASHBOARD

In [50]:
if page=="📊 Dashboard":
    # Make a copy to avoid modifying the global df for other pages if needed
    df_dashboard = df.copy()

    # Generate clean_review, predicted_sentiment, and predicted_driver for the dashboard
    df_dashboard["clean_review"] = df_dashboard["review"].apply(clean_text)
    df_dashboard["predicted_sentiment"] = model.predict(df_dashboard["clean_review"])
    df_dashboard["predicted_driver"] = df_dashboard["clean_review"].apply(detect_driver)

    # KPIs
    c1,c2,c3,c4 = st.columns(4)

    c1.metric(
    "Total Reviews",
    len(df_dashboard)
    )

    c2.metric(
    "Average Rating",
    round(df_dashboard["rating"].mean(),2)
    )

    c3.metric(
    "Countries",
    df_dashboard["country"].nunique()
    )

    c4.metric(
    "Categories",
    df_dashboard["product_category"].nunique()
    )

    # Filters
    country = st.multiselect(
    "Country",
    sorted(df_dashboard["country"].unique())
    )

    category = st.multiselect(
    "Category",
    sorted(df_dashboard["product_category"].unique())
    )

    # Filter dataframe
    filtered = df_dashboard.copy()

    if country:
        filtered = filtered[
            filtered["country"]
            .isin(country)
        ]

    if category:
        filtered = filtered[
            filtered["product_category"]
            .isin(category)
        ]

    # Sentiment Distribution Plot
    fig = px.histogram(
    filtered,
    x="predicted_sentiment",
    color="predicted_sentiment",
    title="Sentiment Distribution"
    )

    st.plotly_chart(fig)

    # Sentiment by Product Category Plot
    fig = px.histogram(

    filtered,

    x="product_category",

    color="predicted_sentiment",

    title="Sentiment by Category"

    )

    st.plotly_chart(fig)

    # Sentiment by Country Plot
    fig = px.histogram(

    filtered,

    x="country",

    color="predicted_sentiment",

    title="Sentiment by Country"

    )

    st.plotly_chart(fig)

    # Sentiment Driver Plot
    fig = px.histogram(

    filtered,

    x="sentiment_driver",

    color="predicted_sentiment",

    title="Sentiment Driver"

    )

    st.plotly_chart(fig)

    # Monthly Trend Plot
    fig = px.line(

    filtered,

    x="month_name",

    y="rating",

    color="predicted_sentiment",

    title="Monthly Trend"

    )

    st.plotly_chart(fig)

#####Model Performance

######This page showcases your machine learning results

In [52]:
if page=="📈 Model Performance":

    st.header("Model Performance")

#####Display images

In [54]:
st.image(

"/content/confusion_matrix (1).png"

)

st.image(

"/content/roc_curve (1).png"

)

st.image(

"/content/feature_importance.png"

)

2026-07-09 19:25:48.507 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:25:48.722 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:25:48.722 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:25:48.723 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:25:48.725 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:25:48.925 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:25:48.926 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:25:48.928 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()

In [55]:
if os.path.exists("/content/shap_summary.png"):

    st.image(

        "/content/shap_summary.png"

    )

2026-07-09 19:27:14.886 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:27:15.322 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:27:15.324 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-09 19:27:15.325 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


####ABOUT

In [58]:
if page=="ℹ️ About":

    st.title("About This Project")

    st.markdown("""

## ShopEase Europe Sentiment Analysis

This project analyses multilingual customer reviews using Natural Language Processing and Machine Learning.

### Technologies

- Python
- Pandas
- Scikit-learn
- TF-IDF
- Streamlit
- Plotly
- Power BI

### Developed By

**Godswill Chukwu**

Data Scientist - Bsc,MSc

""")